# Match_view: auditoría y entrenamiento comparativo

Candidatos YOLO11n (ajuste del modelo activo) y YOLO11s (base preentrenada).
**No despliega pesos.** Las etiquetas actuales son principalmente automáticas;
la validación no demuestra precisión frente a anotación humana independiente.
El ZIP comprobado el 17-11-2026 contiene 4.378 imágenes, no las 4.874 etiquetas
actuales: faltan las 496 imágenes `cirbonero_r2_*`. Las ausencias se registran;
no se inventan imágenes ni etiquetas. No se añaden copias del ZIP adicional
sin comprobar su solapamiento.

Selecciona GPU T4. Ejecuta las celdas en orden. Mantén la sesión de Colab activa.
El acceso a Drive debe autorizarlo el propietario. Cada época guarda una copia
recuperable en una carpeta nueva de Drive, incluyendo el baseline original.


In [ ]:
%pip install -q ultralytics pyyaml
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Selecciona una GPU antes de continuar'
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import subprocess, datetime, zipfile, shutil, sys
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK = Path('/content/match_view_' + STAMP)
WORK.mkdir()
REPO = WORK/'repo'
subprocess.run(['git','clone','--depth','1','https://github.com/Josen-IG/Match_view.git',str(REPO)],check=True)
REVISION = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('Source commit:', REVISION)
ZIP_CANDIDATES = [Path('/content/drive/MyDrive/Git Hub/dataset_yolo_propio_images.zip'),
                  Path('/content/drive/MyDrive/match_view_training_images_20261117.zip')]
ZIP = next((p for p in ZIP_CANDIDATES if p.is_file()), None)
assert ZIP is not None, 'El ZIP compartido necesita estar accesible desde Mi unidad; no cambies su visibilidad pública.'
OUT = Path('/content/drive/MyDrive/Match_view_retraining_' + STAMP)
OUT.mkdir()
with zipfile.ZipFile(ZIP) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(WORK/'unpacked')
roots = list((WORK/'unpacked').rglob('images'))
assert len(roots) == 1, roots
sys.path.insert(0,str(REPO))
from retrain_experiment import prepare, run
shutil.copy2(REPO/'retrain_experiment.py',OUT/'retrain_experiment.py')
(OUT/'source_commit.txt').write_text(REVISION)
DATA, AUDIT = prepare(REPO, roots[0], WORK, OUT)
print('Missing images:',len(AUDIT['missing_images']))
print('Clipped boxes:',len(AUDIT['corrections']))
print('Exact duplicates:',len(AUDIT['exact_duplicates']))
print('Shared matches:',AUDIT['shared_matches'])
print('Artifacts:',OUT)


## Entrenamiento real

60 épocas máximas por candidato, early stopping con paciencia 12, semilla 42,
resolución 960 y batch 8. Sobremuestreo x2 de imágenes con balón, solo en train.
Se conserva el split disponible para comparar ambos candidatos con el baseline.
Los duplicados exactos se excluyen dando prioridad a validación; los parecidos
solo se señalan para revisión. El recorte de cajas a la imagen queda registrado.

**Limitación:** ambos splits comparten partidos y las etiquetas no están
exhaustivamente revisadas a mano. La integración requiere un holdout independiente
anotado y revisado por personas, además de una mejora del recall del balón sin
una pérdida importante de precisión y un coste de CPU aceptable.


In [ ]:
import json, traceback
try:
    run(REPO, DATA, OUT, epochs=60)
except Exception:
    (OUT/'failure.txt').write_text(traceback.format_exc())
    raise
print((OUT/'status.json').read_text())
print('El modelo de producción sigue intacto.')


## Artefactos y lectura de métricas

En `OUT`: `audit.json`, hashes y manifest, correcciones, etiquetas ausentes,
versiones de paquetes, copia del código, baseline original y carpetas por candidato
con `best.pt`, `last.pt`, `resumable.pt`, `results.csv`, argumentos y gráficos.

Cada modelo tiene métricas AP por clase; precisión/recall y FP a confianza 0.25,
IoU de emparejamiento 0.5 y NMS IoU 0.7. El recall de objetos diminutos usa el lado
mayor de la caja <=10 píxeles tras escalar el lado mayor de la imagen a 960.
No mezcles precision/recall al mejor F1 con precision/recall a umbral fijo.

CPU: mismos 40 fotogramas distribuidos por validación, 2 hilos, 3 calentamientos,
tiempo completo de predicción excepto lectura de imagen. Es la CPU de Colab;
hay que confirmar el coste en el equipo Windows antes del despliegue.

Las predicciones de Kalman/interpolación y los eventos experimentales de Match_view
no intervienen en la puntuación de detección. Este cuaderno no modifica la interfaz.
